# Celebal Excellence Internship (CEI) 2026
## Data Engineering Domain — Week 5 Assignment
### Data Cleaning, Transformation & Aggregation with PySpark

---

| Field | Detail |
|---|---|
| **Intern Name** | *Himanshu Batra* |
| **Internship** | Celebal Excellence Internship (CEI) 2026 |
| **Domain** | Data Engineering |
| **Week** | Week 5 |
| **Registration / Intern ID** | *CT_CSI_DE_1103* |
| **Tool Stack** | PySpark, Google Colab, Python 3 |


## Objective

This notebook addresses the Week 5 assignment on **large-scale data cleaning and transformation using PySpark**. It covers foundational Spark theory (MapReduce limitations, in-memory computing, DataFrame immutability, shuffles) alongside hands-on data engineering tasks — deduplication, null handling, filtering, type casting, and multi-stage aggregation pipelines.

Rather than instantiating a new toy DataFrame for every question, a **single realistic Superstore-style transactional dataset** is built once and reused throughout, mirroring how a data engineer would work against one production table across a notebook.

## Technologies Used

- **Apache Spark (PySpark)** — distributed DataFrame processing engine
- **Google Colab** — execution environment
- **Python 3** — driver language
- **Pandas** — used only to seed the initial synthetic dataset before conversion to a Spark DataFrame


## 1. Environment Setup

### 1.1 Install PySpark

Google Colab does not ship with Spark pre-installed, so it is installed at runtime. Pinning a specific version keeps the notebook reproducible across sessions.

In [17]:
!pip install -q pyspark==4.0.0


### 1.2 Import Libraries

In [18]:
import random
from datetime import datetime, timedelta

import pandas as pd
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType

print("PySpark version:", pyspark.__version__)


PySpark version: 3.5.1


### 1.3 Create Spark Session

A local Spark session is sufficient for this notebook's dataset size. `spark.sql.shuffle.partitions` is lowered from its default of 200, since a small local dataset does not benefit from that many shuffle partitions.

In [19]:
spark = (
    SparkSession.builder
    .appName("CEI-Week5-DataEngineering-Superstore")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark


## 2. Dataset

A single, consistent dataset is used for **every** question in this notebook. It is modeled on the well-known **Superstore** retail dataset (`region`, `city`, `product_category`, `sale_amount`) and extended with the additional columns the assignment questions require (`status`, `age`, `subscription`, `raw_timestamp`, `username`, `email`, `store_id`, `price`, `user_id`, `transaction_date`).

The data is generated with a fixed random seed so results are fully reproducible, and it intentionally contains:
- **duplicate `(user_id, transaction_date)` pairs** — for the deduplication tasks (Q3, Q15)
- **null values** in `status`, `email`, `price` and **empty strings** in `username` — for the null-handling tasks (Q5, Q9, Q12, Q13, Q15)
- **string-formatted timestamps** in `raw_timestamp` — for the type-casting task (Q10)

No external file download is required — the dataset is built entirely in-memory.

In [20]:
random.seed(42)

REGIONS = ["West", "East", "Central", "South"]
CATEGORIES = ["Furniture", "Office Supplies", "Technology"]
CITIES = [
    "New York", "Los Angeles", "Chicago", "Houston", "Phoenix",
    "Philadelphia", "San Antonio", "San Diego", "Dallas", "Austin",
]
STATUS_VALUES = ["Shipped", "Delivered", "Pending", "Cancelled", None]
SUBSCRIPTION_TIERS = ["Premium", "Standard", "Basic"]

COLUMNS = [
    "order_id", "user_id", "transaction_date", "region", "city",
    "product_category", "sale_amount", "status", "age", "subscription",
    "raw_timestamp", "username", "email", "store_id", "price",
]


def generate_superstore_records(n_records: int = 1200):
    """Builds a realistic, extended Superstore-style transaction record set."""
    records = []
    for i in range(1, n_records + 1):
        user_id = f"U{random.randint(1000, 1199)}"
        order_date = datetime(2023, 1, 1) + timedelta(days=random.randint(0, 720))
        region = random.choice(REGIONS)
        city = random.choice(CITIES)
        category = random.choice(CATEGORIES)
        sale_amount = round(random.uniform(15, 2500), 2)
        status = random.choice(STATUS_VALUES)
        age = random.randint(16, 65)
        subscription = random.choice(SUBSCRIPTION_TIERS)
        raw_timestamp = order_date.strftime("%Y-%m-%d %H:%M:%S")
        username = f"user_{user_id.lower()}" if random.random() > 0.03 else ""
        email = f"{user_id.lower()}@example.com" if random.random() > 0.06 else None
        store_id = f"ST-{random.randint(1, 20):03d}"
        price = sale_amount if random.random() > 0.05 else None

        records.append((
            f"ORD-{i:05d}", user_id, order_date.strftime("%Y-%m-%d"), region, city,
            category, sale_amount, status, age, subscription, raw_timestamp,
            username, email, store_id, price,
        ))
    return records


records = generate_superstore_records(1200)
pdf = pd.DataFrame(records, columns=COLUMNS)

# Deliberately re-insert a sample of rows to create duplicate (user_id, transaction_date)
# pairs, so the deduplication logic in Q3 / Q15 has something real to remove.
duplicate_sample = pdf.sample(60, random_state=7)
pdf = pd.concat([pdf, duplicate_sample], ignore_index=True)

df = spark.createDataFrame(pdf)
df.cache()

print("Total records loaded:", df.count())


Total records loaded: 1260


### 2.1 Dataset Preview

In [21]:
df.show(10, truncate=False)


+---------+-------+----------------+-------+------------+----------------+-----------+---------+---+------------+-------------------+----------+-----------------+--------+-------+
|order_id |user_id|transaction_date|region |city        |product_category|sale_amount|status   |age|subscription|raw_timestamp      |username  |email            |store_id|price  |
+---------+-------+----------------+-------+------------+----------------+-----------+---------+---+------------+-------------------+----------+-----------------+--------+-------+
|ORD-00001|U1163  |2023-04-25      |West   |Phoenix     |Furniture       |569.68     |Shipped  |59 |Basic       |2023-04-25 00:00:00|user_u1163|u1163@example.com|ST-014  |NaN    |
|ORD-00002|U1023  |2023-08-12      |East   |Dallas      |Technology      |80.94      |Delivered|61 |Basic       |2023-08-12 00:00:00|user_u1023|u1023@example.com|ST-015  |80.94  |
|ORD-00003|U1001  |2023-06-13      |South  |Philadelphia|Office Supplies |401.37     |Pending  |22 |

### 2.2 Dataset Schema

In [22]:
df.printSchema()


root
 |-- order_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- transaction_date: string (nullable = true)
 |-- region: string (nullable = true)
 |-- city: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- age: long (nullable = true)
 |-- subscription: string (nullable = true)
 |-- raw_timestamp: string (nullable = true)
 |-- username: string (nullable = true)
 |-- email: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- price: double (nullable = true)



## 3. Questions Q1 – Q15

### Q1. Limitations of Traditional MapReduce

**Task:** Explain the key limitations of traditional MapReduce that make Spark the preferred choice for modern big data processing.

Traditional MapReduce, while foundational to distributed batch processing, has several limitations that make it unsuitable for many modern workloads. Every MapReduce job persists intermediate results to disk between the Map and Reduce phases, which introduces heavy I/O overhead and makes iterative algorithms — such as those used in machine learning — extremely slow, since each iteration re-reads and re-writes data from disk. The programming model is also rigid: expressing anything beyond a simple map-then-reduce flow (joins, multi-stage pipelines, graph algorithms) requires chaining multiple jobs manually, which increases complexity and job-scheduling overhead. MapReduce also lacks built-in support for real-time or near-real-time processing and offers limited interactive query capability, with high per-job startup latency. Spark addresses these gaps through in-memory computation, a richer DAG-based execution engine, and unified APIs spanning batch, streaming, SQL, and machine learning — resulting in substantially faster and more flexible data processing.

### Q2. In-Memory Computing in Spark

**Task:** Explain how Spark uses in-memory computing to speed up iterative machine learning algorithms compared to disk-based systems.

Spark's performance advantage over disk-based engines like MapReduce comes primarily from in-memory computing. Instead of writing intermediate results to disk after every stage, Spark keeps data in RAM across operations using Resilient Distributed Datasets (RDDs) or DataFrames, and allows a dataset that is reused multiple times to be explicitly cached or persisted. This is especially valuable for iterative machine learning algorithms — such as gradient descent, k-means, or ALS — which repeatedly scan the same dataset across many iterations. By caching the dataset in memory once, subsequent iterations read directly from RAM instead of performing costly disk I/O, cutting execution time from hours to minutes in many cases. Spark's DAG scheduler also optimizes the execution plan across an entire pipeline rather than treating each stage as an isolated job, further reducing redundant computation and unnecessary disk access.

### Q3. Removing Duplicate Rows

**Task:** Remove all duplicate rows from the DataFrame based on a specific set of columns: `user_id` and `transaction_date`.

`dropDuplicates()` is used with an explicit column subset so that Spark only considers `user_id` and `transaction_date` when deciding whether two rows are duplicates — other columns are ignored for the comparison, and the first occurrence encountered per partition is retained.

In [23]:
df_dedup = df.dropDuplicates(["user_id", "transaction_date"])

print("Row count before deduplication:", df.count())
print("Row count after deduplication:", df_dedup.count())
df_dedup.select("order_id", "user_id", "transaction_date").show(10)


Row count before deduplication: 1260
Row count after deduplication: 1196
+---------+-------+----------------+
| order_id|user_id|transaction_date|
+---------+-------+----------------+
|ORD-00330|  U1000|      2023-04-04|
|ORD-00756|  U1000|      2024-04-12|
|ORD-00003|  U1001|      2023-06-13|
|ORD-00750|  U1001|      2023-06-22|
|ORD-00416|  U1002|      2023-02-13|
|ORD-00797|  U1002|      2023-04-14|
|ORD-00637|  U1002|      2023-04-16|
|ORD-00824|  U1002|      2023-04-19|
|ORD-01025|  U1002|      2023-06-12|
|ORD-00842|  U1002|      2023-11-13|
+---------+-------+----------------+
only showing top 10 rows



**Output explanation:** The row count drops after deduplication, confirming that the intentionally duplicated `(user_id, transaction_date)` pairs injected during dataset creation were correctly identified and removed. Every remaining `(user_id, transaction_date)` combination in `df_dedup` is now unique.

### Q4. Average Sales by Category in the West Region

**Task:** Filter `df` for rows where `region = 'West'`, then group by `product_category` to find the average `sale_amount`.


In [24]:
df_west_avg = (
    df.filter(F.col("region") == "West")
      .groupBy("product_category")
      .agg(F.round(F.avg("sale_amount"), 2).alias("avg_sale_amount"))
      .orderBy(F.col("avg_sale_amount").desc())
)

df_west_avg.show()


+----------------+---------------+
|product_category|avg_sale_amount|
+----------------+---------------+
|      Technology|        1267.05|
| Office Supplies|        1252.28|
|       Furniture|        1165.54|
+----------------+---------------+



**Output explanation:** The result shows one row per `product_category` present in the West region, each with its average `sale_amount` rounded to two decimal places, ordered from highest to lowest average sale. This kind of regional/category breakdown is a common first step in sales performance analysis.

### Q5. `.na.drop()` vs `.na.fill()`

**Task:** Explain the difference between `.na.drop()` and `.na.fill()`, and fill null values in the `status` column with `'Unknown'`.

`.na.drop()` and `.na.fill()` serve opposite purposes when handling missing data. `.na.drop()` **removes** rows containing null values — by default any row with at least one null, though it can be restricted to specific columns or a minimum non-null threshold using its `how` and `subset` parameters. `.na.fill()` instead **replaces** nulls with a specified value, either applied uniformly across all compatible columns or targeted per column using a dictionary. The right choice depends on whether missing data is safe to discard entirely (`drop`) or should be preserved with a sensible default so downstream aggregations and row counts aren't skewed by row loss (`fill`).

In [25]:
df_status_filled = df.na.fill({"status": "Unknown"})

print("Null status count before fill:", df.filter(F.col("status").isNull()).count())
print("Null status count after fill:", df_status_filled.filter(F.col("status").isNull()).count())
df_status_filled.select("order_id", "status").show(10)


Null status count before fill: 268
Null status count after fill: 0
+---------+---------+
| order_id|   status|
+---------+---------+
|ORD-00001|  Shipped|
|ORD-00002|Delivered|
|ORD-00003|  Pending|
|ORD-00004|  Unknown|
|ORD-00005|  Pending|
|ORD-00006|  Unknown|
|ORD-00007|  Unknown|
|ORD-00008|Cancelled|
|ORD-00009|  Shipped|
|ORD-00010|  Pending|
+---------+---------+
only showing top 10 rows



**Output explanation:** After the fill, the null count for `status` drops to zero — every previously null value now reads `'Unknown'`, while all other columns and non-null `status` values are left untouched.

### Q6. City Record Counts Above a Threshold

**Task:** Find the total count of records for each `city`, but only return cities where the count is greater than 100.


In [26]:
df_city_counts = (
    df.groupBy("city")
      .agg(F.count("*").alias("record_count"))
      .filter(F.col("record_count") > 100)
      .orderBy(F.col("record_count").desc())
)

df_city_counts.show()


+------------+------------+
|        city|record_count|
+------------+------------+
|      Dallas|         134|
| San Antonio|         133|
|     Phoenix|         130|
|     Chicago|         129|
|      Austin|         129|
|   San Diego|         127|
| Los Angeles|         127|
|Philadelphia|         126|
|     Houston|         118|
|    New York|         107|
+------------+------------+



**Output explanation:** `groupBy` + `count` produces the per-city totals, and the subsequent `filter` is applied *after* aggregation (equivalent to a SQL `HAVING` clause) so only cities clearing the 100-record threshold appear in the final result.

### Q7. DataFrame Immutability and Data Cleaning

**Task:** How does the immutability of Spark DataFrames affect how you perform data cleaning steps like dropping or renaming columns?

Spark DataFrames are immutable — once created, their contents cannot be changed in place. Any transformation, such as dropping a column, renaming a column, or filtering rows, does not modify the original DataFrame; instead, it returns a **brand-new DataFrame** while the original remains untouched. This has direct implications for data cleaning workflows: cleaning steps must be expressed as a chain of transformations (for example, `df2 = df.drop('col').withColumnRenamed('a', 'b')`) rather than as in-place edits, and each intermediate result needs to be reassigned to a variable if it will be reused later. While this can feel unfamiliar coming from pandas, immutability is what enables Spark's lazy evaluation and lineage tracking, which in turn underpin fault tolerance — if a partition is lost, Spark can recompute it from the original DataFrame using the recorded chain of transformations.

### Q8. Filtering Premium Subscribers Aged 18–30

**Task:** Filter the dataset for rows where `age` is between 18 and 30 (inclusive) and `subscription = 'Premium'`.


In [27]:
df_young_premium = df.filter(
    (F.col("age").between(18, 30)) & (F.col("subscription") == "Premium")
)

print("Matching records:", df_young_premium.count())
df_young_premium.select("user_id", "age", "subscription", "city").show(10)


Matching records: 115
+-------+---+------------+------------+
|user_id|age|subscription|        city|
+-------+---+------------+------------+
|  U1001| 22|     Premium|Philadelphia|
|  U1150| 30|     Premium|      Austin|
|  U1009| 30|     Premium|     Houston|
|  U1122| 25|     Premium| San Antonio|
|  U1065| 26|     Premium|    New York|
|  U1156| 23|     Premium|   San Diego|
|  U1011| 29|     Premium|      Austin|
|  U1040| 24|     Premium|Philadelphia|
|  U1039| 21|     Premium|      Austin|
|  U1152| 23|     Premium|      Dallas|
+-------+---+------------+------------+
only showing top 10 rows



**Output explanation:** `.between(18, 30)` applies an inclusive range filter, combined with an equality check on `subscription` using `&`. The result is the subset of premium-tier users in the 18–30 age band, useful for targeted marketing or cohort analysis.

### Q9. Handling Nulls Before Aggregation

**Task:** When cleaning a dataset, why is it often better to handle null values before performing mathematical aggregations like `sum()` or `avg()`?

Null values can silently distort the results of aggregation functions like `sum()` and `avg()`. Spark's aggregate functions ignore nulls by default, meaning a column with several nulls produces an average calculated only over the non-null rows — giving a misleadingly optimistic or skewed figure compared to what the business actually expects if nulls should have counted as zero or been imputed. Nulls can also propagate through derived calculations, since a null in one column used inside an arithmetic expression makes the entire computed result null, silently dropping rows from downstream logic. Handling nulls explicitly — by dropping, imputing, or flagging them — before aggregating ensures the resulting metrics accurately reflect the true dataset and avoids introducing invisible bias into reports and downstream models.

### Q10. Casting `raw_timestamp` to `TimestampType`

**Task:** Revise the `raw_timestamp` column by casting it to `TimestampType` and renaming it to `event_time`.


In [28]:
df_event_time = (
    df.withColumn("raw_timestamp", F.col("raw_timestamp").cast(TimestampType()))
      .withColumnRenamed("raw_timestamp", "event_time")
)

df_event_time.select("order_id", "event_time").printSchema()
df_event_time.select("order_id", "event_time").show(5, truncate=False)


root
 |-- order_id: string (nullable = true)
 |-- event_time: timestamp (nullable = true)

+---------+-------------------+
|order_id |event_time         |
+---------+-------------------+
|ORD-00001|2023-04-25 00:00:00|
|ORD-00002|2023-08-12 00:00:00|
|ORD-00003|2023-06-13 00:00:00|
|ORD-00004|2024-04-15 00:00:00|
|ORD-00005|2023-08-27 00:00:00|
+---------+-------------------+
only showing top 5 rows



**Output explanation:** The schema output confirms `event_time` is now `timestamp` type rather than `string`. Since `raw_timestamp` was generated in `yyyy-MM-dd HH:mm:ss` format, Spark's default cast parses it directly; for less standard formats, `F.to_timestamp(col, format)` should be used instead to control parsing explicitly.

### Q11. Shuffle and Wide Transformations

**Task:** Explain the "Shuffle" process that occurs during a grouping operation, and why it is considered a wide transformation.

A shuffle is the process of redistributing data across partitions — and often across executors and the network — so that records sharing the same key end up on the same partition. It happens whenever an operation needs to combine data that isn't already co-located, such as `groupBy`, `join`, or `repartition`. Because `groupBy` requires all rows sharing a key to be brought together before aggregation, it triggers a shuffle; this is why it is classified as a **wide transformation**, where each output partition can depend on data from multiple input partitions. In contrast, **narrow transformations** like `filter` or `select` operate independently within each partition without any cross-partition data movement. Shuffles are expensive because they involve disk I/O, network transfer, and serialization/deserialization, making them one of the primary targets for performance tuning in Spark jobs.

### Q12. Removing Rows with Null Email or Empty Username

**Task:** Identify and remove rows where `email` is null **or** `username` is an empty string.


In [29]:
df_clean_contacts = df.filter(
    F.col("email").isNotNull() & (F.col("username") != "")
)

print("Rows before cleaning:", df.count())
print("Rows after removing null email / empty username:", df_clean_contacts.count())
df_clean_contacts.select("order_id", "username", "email").show(10)


Rows before cleaning: 1260
Rows after removing null email / empty username: 1159
+---------+----------+-----------------+
| order_id|  username|            email|
+---------+----------+-----------------+
|ORD-00001|user_u1163|u1163@example.com|
|ORD-00002|user_u1023|u1023@example.com|
|ORD-00003|user_u1001|u1001@example.com|
|ORD-00004|user_u1011|u1011@example.com|
|ORD-00005|user_u1020|u1020@example.com|
|ORD-00006|user_u1043|u1043@example.com|
|ORD-00007|user_u1008|u1008@example.com|
|ORD-00008|user_u1035|u1035@example.com|
|ORD-00009|user_u1012|u1012@example.com|
|ORD-00010|user_u1002|u1002@example.com|
+---------+----------+-----------------+
only showing top 10 rows



**Output explanation:** The filter keeps only rows that satisfy the *inverse* condition of what should be removed — `email` is not null **and** `username` is not an empty string — which is logically equivalent to dropping every row where `email IS NULL OR username = ''`. The row count decreases accordingly.

### Q13. Multiple Aggregations with `.agg()`

**Task:** Use `.agg()` to calculate the minimum, maximum, and mean of the `price` column in a single pass.


In [30]:
df_price_stats = df.agg(
    F.min("price").alias("min_price"),
    F.max("price").alias("max_price"),
    F.round(F.avg("price"), 2).alias("avg_price"),
)

df_price_stats.show()


+---------+---------+---------+
|min_price|max_price|avg_price|
+---------+---------+---------+
|    18.63|      NaN|      NaN|
+---------+---------+---------+



**Output explanation:** A single `.agg()` call with multiple aggregate expressions computes all three statistics in one Spark job rather than three separate passes over the data, which is both more concise and more efficient than calling `.min()`, `.max()`, and `.avg()` independently.

### Q14. Risks of `inferSchema=True`

**Task:** In the context of cleaning a dataset, what is the risk of using `inferSchema=True` when the source data contains messy or inconsistent date formats?

Setting `inferSchema=True` tells Spark to scan the source data and automatically infer each column's data type, which is convenient but risky for messy or inconsistent data. Spark infers types based on a sample or full pass over the data, and if a date column contains mixed formats — for example `'2023-01-05'` alongside `'01/05/2023'` or stray non-date text — Spark may fall back to inferring the column as `StringType` instead of `DateType`/`TimestampType`, silently masking a data quality problem instead of surfacing it. Schema inference also adds an extra read pass over the file, increasing load time on large datasets. In production pipelines, it is generally safer to explicitly define a `StructType` schema: this guarantees consistent types, fails fast on genuinely malformed records, and avoids subtle downstream bugs caused by incorrect type inference on edge cases.

### Q15. End-to-End Processing Pipeline

**Task:** Build one complete processing pipeline that: removes duplicates, fills null `price` values with `0`, groups by `store_id`, and calculates total revenue.

This mirrors a realistic cleaning-to-reporting pipeline: deduplicate on the natural key, neutralize nulls so they don't silently drop out of the sum, then aggregate to a business metric — total revenue per store.

In [31]:
df_pipeline = (
    df.dropDuplicates(["user_id", "transaction_date"])
      .na.fill({"price": 0})
      .groupBy("store_id")
      .agg(F.round(F.sum("price"), 2).alias("total_revenue"))
      .orderBy(F.col("total_revenue").desc())
)

df_pipeline.show(20)


+--------+-------------+
|store_id|total_revenue|
+--------+-------------+
|  ST-017|     91965.28|
|  ST-014|     91751.13|
|  ST-011|     89419.29|
|  ST-010|     87732.67|
|  ST-003|     85428.99|
|  ST-018|     84506.46|
|  ST-008|      81096.6|
|  ST-004|     79591.61|
|  ST-016|     77660.52|
|  ST-005|     75019.43|
|  ST-019|     73043.85|
|  ST-001|     71923.88|
|  ST-013|     71428.55|
|  ST-012|     64170.73|
|  ST-006|     60983.51|
|  ST-002|     59643.26|
|  ST-007|     54992.51|
|  ST-015|     51645.33|
|  ST-009|     50958.21|
|  ST-020|     48133.38|
+--------+-------------+



**Output explanation:** The pipeline chains four operations lazily — Spark only executes the full DAG when `.show()` is called. The output is one row per `store_id` with its total revenue, computed only after duplicate transactions were removed and null prices were neutralized to zero rather than excluded, so no store's revenue is understated due to missing price data.

## 4. Learning Outcomes & Key Concepts

**Learning Outcomes**
- Built and reasoned about a single, realistic PySpark DataFrame across an entire multi-question workflow, instead of isolated toy examples.
- Applied deduplication, null-handling, filtering, type casting, and multi-column aggregation using idiomatic PySpark DataFrame APIs.
- Practiced translating a plain-language cleaning requirement into a chained, lazily-evaluated Spark transformation pipeline.

**Key Concepts Learned**
- **In-memory computing** and why it outperforms disk-based MapReduce for iterative workloads.
- **DataFrame immutability** and its effect on how cleaning steps must be chained rather than applied in place.
- **Wide vs. narrow transformations**, and why grouping/joining operations trigger an expensive shuffle.
- **Null-handling strategy** (`.na.drop()` vs `.na.fill()`) and why it must precede aggregation.
- The risk explicit schemas mitigate versus relying on `inferSchema=True` on messy source data.


## 5. Conclusion

This notebook worked through the full Week 5 syllabus — from core Spark theory to hands-on DataFrame engineering — using one consistent, realistic dataset throughout. The combination of conceptual explanations and executable PySpark code reflects how a data engineer would actually approach a messy transactional dataset: understand *why* a technique is needed, then implement it cleanly using the DataFrame API. The resulting pipeline (Q15) ties every individual technique — deduplication, null handling, and aggregation — together into a single production-style transformation.